### Document Indexing with PyPDF

In [1]:
import copy

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter

/tmp/ipykernel_303/4098098794.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
loader_pdf = PyPDFLoader("files/Introduction_to_Data_and_Data_Science.pdf")
pages_pdf = loader_pdf.load()
page_num = len(pages_pdf)  # 6 pages
page_info = pages_pdf[4].metadata

In [3]:
# Remove \n new line characters as they consume tokens
pages = copy.deepcopy(pages_pdf)
pdf_page_1 = pages[0].page_content.split() # Makes a list and removes the \n chars
pdf_page_1 = " ".join(pdf_page_1)
pdf_page_1

'Analysis vs Analytics Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both. So, let’s clear this up, shall we? First, we will start with analysis. Consider the following… You have a huge dataset containing data of various types. Instead of tackling the entire dataset and running the risk of becoming overwhelmed, you separate it into easier to digest chunks and study them individually and examine how they relate to other parts. And that’s analysis in a nutshell. One important thing to remember, however, is that you perform analyses on things that have already happened in the past. Such as using an analysis to explain how a

Checking tokenizer count in open AI, the first page is 421 with \n and without \n, it's 306 tokens.. a huge reduction.. think of all six pages.

In [4]:
for page in pages:
    page.page_content = " ".join(page.page_content.split())  # Actually overwrites pages_pdf content

In [5]:
pages

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2023-11-09T10:16:34+02:00', 'author': 'Hristina  Hristova', 'moddate': '2023-11-09T10:16:34+02:00', 'source': 'files/Introduction_to_Data_and_Data_Science.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='Analysis vs Analytics Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both. So, let’s clear this up, shall we? First, we will start with analysis. Consider the following… You have a huge dataset containing data of various types. Instead of tackling the entire dataset and 

In [22]:
char_splitter = CharacterTextSplitter(separator=".", chunk_size=500, chunk_overlap=0)
pages_split = char_splitter.split_documents(pages)
pages_split[1].page_content

'So, let’s clear this up, shall we? First, we will start with analysis. Consider the following… You have a huge dataset containing data of various types. Instead of tackling the entire dataset and running the risk of becoming overwhelmed, you separate it into easier to digest chunks and study them individually and examine how they relate to other parts. And that’s analysis in a nutshell'

### Indexing with DocLoader and MarkDown Splitter

In [31]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters.markdown import MarkdownHeaderTextSplitter

loader_docx = Docx2txtLoader("files/Introduction_to_Data_and_Data_Science.docx")
pages = loader_docx.load()

split_on = [("#", "Course Title"), ("##", "Lecture Title")]
md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=split_on)
# Length is 1 page according to the loader, so we fetch it and split
pages_split = md_splitter.split_text((pages[0]).page_content)
print(pages_split[0].metadata, "\n", pages_split[1].metadata)

{'Course Title': 'Introduction to Splitting', 'Lecture Title': 'Analysis vs Analytics Alright!'} 
 {'Course Title': 'Introduction to Splitting', 'Lecture Title': 'Programming Languages & Software Employed in Data Science -'}


### Text Embedding with OpenAI

In [36]:
pages = copy.deepcopy(pages_split)
for num in range(len(pages)):
    pages[num].page_content = " ".join(pages[num].page_content.split())

pages_char_split = char_splitter.split_documents(pages)
pages_char_split

[Document(metadata={'Course Title': 'Introduction to Splitting', 'Lecture Title': 'Analysis vs Analytics Alright!'}, page_content='So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both. So, let’s clear this up, shall we? First, we will start with analysis'),
 Document(metadata={'Course Title': 'Introduction to Splitting', 'Lecture Title': 'Analysis vs Analytics Alright!'}, page_content='Consider the following… You have a huge dataset containing data of various types. Instead of tackling the entire dataset and running the risk of becoming overwhelmed, you separate it into easier to digest chunks and study them individually and examin

In [40]:
from langchain_openai.embeddings import OpenAIEmbeddings

from config import OPEN_AI_KEY as API_KEY

embedding = OpenAIEmbeddings(model="text-embedding-3-small", api_key=API_KEY)
vector_1 = embedding.embed_query(pages_char_split[3].page_content)
vector_2 = embedding.embed_query(pages_char_split[5].page_content)
vector_3 = embedding.embed_query(pages_char_split[18].page_content)

In [49]:
import numpy as np
# Compare how close they are to each other
print(np.dot(vector_1, vector_2), np.dot(vector_1, vector_3), np.dot(vector_2, vector_3))
print(np.linalg.norm(vector_1), np.linalg.norm(vector_2), np.linalg.norm(vector_3))

0.5469643853602975 0.3994167640659043 0.34540798791346106
1.0000864483738017 0.9999574364400718 0.9997614304560071


### Creating a Chroma Vectorstore

In [50]:
from langchain_community.vectorstores import Chroma

# Vector store to keep all 20 docs and their vector representation
vector_store = Chroma.from_documents(documents=pages_char_split, embedding=embedding, persist_directory="./files")

In [53]:
# Get saved vectors
vector_store_from_directory = Chroma(persist_directory="./files", embedding_function=embedding)

/tmp/ipykernel_303/373466818.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store_from_directory = Chroma(persist_directory="./files", embedding_function=embedding)


In [55]:
vector_store_from_directory.get()

{'ids': ['b982187d-e31d-492c-b8cf-4b990f37480e',
  '84cd489a-77f9-4197-a168-d2a9fa1a9556',
  'b4d2468f-202c-4e5d-96eb-134ddc87c3cc',
  '5eab28fa-acfe-4006-8862-58d2516b0017',
  '2ac18e82-96bd-4706-a691-23c46bce1c13',
  'bb581a43-e078-4d6a-94e2-a96fa1096152',
  '3f9951ac-3ef3-49b0-a50a-0626201765e6',
  'e711081c-74a4-4cbc-baab-d5f08fba1128',
  'ad6b5d18-94e1-4363-bf0b-38365c52cf64',
  '8865561f-79a7-4692-8f53-e9d86404ef99',
  '5bd80420-7601-4291-8385-2f60307ad140',
  'fcce9c69-44e7-4aca-9acd-c1e261275134',
  'c022f2d3-f892-4047-968e-11613243f78c',
  '26ffbece-cf4a-40eb-8725-7235e8e0b495',
  '0434f1a2-c2be-4a62-8514-6e94ad68e5ea',
  'fa9e9905-0d6f-4556-a6fd-c606aae83077',
  '89e27b26-4488-4366-9725-db564bc7bbd8',
  '35ec85c9-6338-4fd4-92b5-75afb946bb5c',
  'd1cb38f2-4275-4c69-a06a-5b8eb0386122',
  '689d0200-272e-4793-a383-d3920880956f'],
 'embeddings': None,
 'documents': ['So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of

In [64]:
vector_store.get(ids="5eab28fa-acfe-4006-8862-58d2516b0017", include=["embeddings", "documents"])

{'ids': ['5eab28fa-acfe-4006-8862-58d2516b0017'],
 'embeddings': array([[-0.02207947,  0.03869629,  0.05041504, ..., -0.01121521,
          0.00782776,  0.01707458]]),
 'documents': ['Analytics is essentially the application of logical and computational reasoning to the component parts obtained in an analysis. And in doing this you are looking for patterns and exploring what you could do with them in the future. Here, analytics branches off into two areas: qualitative analytics – this is using your intuition and experience in conjunction with the analysis to plan your next business move'],
 'uris': None,
 'included': ['embeddings', 'documents'],
 'data': None,
 'metadatas': None}

4937

In [71]:
# Let's add a new document to the stored vector
from langchain_core.documents import Document
random_text = """
Yes, that is completely normal and expected.
Those values are effectively 1.0.
The tiny variations (like 1.000086 or 0.999957) are just negligible floating-point precision differences from 32-bit/64-bit numerical calculations during embedding generation and NumPy operations.
Because they are so close to 1, you can confidently treat them as normalized unit vectors, meaning np.dot() will accurately give you the cosine similarity score every time.
"""
added_document = Document(page_content=random_text, metadata={"Course Title": "Introduction to Data", "Lecture Title": "Embedding"})
vector_store_from_directory.add_documents([added_document])

['4dcbf363-9354-4b95-9c16-a8d217e54a9b']

In [77]:
vector_store.get("4dcbf363-9354-4b95-9c16-a8d217e54a9b")   # The two vectors are all connected to the same database, that's why it gets the added document

{'ids': ['4dcbf363-9354-4b95-9c16-a8d217e54a9b'],
 'embeddings': None,
 'documents': ['\nYes, that is completely normal and expected.\nThose values are effectively 1.0.\nThe tiny variations (like 1.000086 or 0.999957) are just negligible floating-point precision differences from 32-bit/64-bit numerical calculations during embedding generation and NumPy operations.\nBecause they are so close to 1, you can confidently treat them as normalized unit vectors, meaning np.dot() will accurately give you the cosine similarity score every time.\n'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'Lecture Title': 'Embedding',
   'Course Title': 'Introduction to Data'}]}

In [78]:
# Let's update
random_text = """
Updating a document in a vector store requires an ID so Chroma knows which entry to overwrite instead of creating a duplicate. When you pass an updated document with an existing ID using update_documents(), Chroma automatically generates a new vector embedding for the new text and replaces both the old vector and its metadata in your persistent database.
"""
updated_document = Document(page_content=random_text, metadata={"Course Title": "Introduction to Data", "Lecture Title": "Updating Vector Store"})
vector_store_from_directory.update_document(document_id="4dcbf363-9354-4b95-9c16-a8d217e54a9b", document=updated_document)

In [79]:
vector_store.get("4dcbf363-9354-4b95-9c16-a8d217e54a9b")

{'ids': ['4dcbf363-9354-4b95-9c16-a8d217e54a9b'],
 'embeddings': None,
 'documents': ['\nUpdating a document in a vector store requires an ID so Chroma knows which entry to overwrite instead of creating a duplicate. When you pass an updated document with an existing ID using update_documents(), Chroma automatically generates a new vector embedding for the new text and replaces both the old vector and its metadata in your persistent database.\n'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'Lecture Title': 'Updating Vector Store',
   'Course Title': 'Introduction to Data'}]}

In [84]:
vector_store.delete("4dcbf363-9354-4b95-9c16-a8d217e54a9b")
vector_store.get("4dcbf363-9354-4b95-9c16-a8d217e54a9b")

{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}